In [6]:
from image_processing_utils import process_image, harsh_gamma_enhance, normalize, erode, resize_symbols, resize_image
from utils import load_images
from torchvision import transforms
import torch
import cv2
import numpy as np

import itertools
import io
import base64
from PIL import Image

np.set_printoptions(
    threshold=np.inf,
    linewidth=np.inf,
    precision=2,
    suppress=True
)

torch.set_printoptions(threshold=float('inf'), linewidth=400, precision=3)

label_decoding = {
    10: '(',
    11: ')',
    12: '+',
    13: '-',
    14: '=',
    15: 'fwd_slash',
    16: 'times'
}

In [7]:
resizing_methods = [
    transforms.Resize((28, 28)),
    transforms.Lambda(lambda image: resize_symbols(image, mode="pad")),
    transforms.Lambda(lambda image: resize_symbols(image, mode="stretch"))
]

transformation_methods = [
    transforms.Lambda(erode),
    transforms.Lambda(harsh_gamma_enhance)
]

# transformation pipeline
resize_after_transform = transforms.Compose([
    transforms.Lambda(erode),
    transforms.Lambda(harsh_gamma_enhance),
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

resize_before_transform = transforms.Compose([
    transforms.Lambda(resize_image),
    transforms.Lambda(harsh_gamma_enhance),
    transforms.Lambda(erode),
    transforms.Lambda(lambda image: resize_symbols(image, mode="stretch")),
    transforms.Lambda(normalize),
    transforms.ToTensor()
])

with open("./data/3+3=.txt", "r") as file:
    image = Image.open(io.BytesIO(base64.decodebytes(bytes(file.read(), "utf-8"))))

image_data = load_images(s3_bucket_path="data/token/=/", s3_bucket_name="penman-lln")[0]


Obtaining images...: 219it [00:22,  9.74it/s]


In [ ]:
resizing_methods = [
    (transforms.Resize((28, 28)), "pytorch_resize"),
    (transforms.Lambda(lambda image: resize_symbols(image, mode="pad")), "default_pad"),
    (transforms.Lambda(lambda image: resize_symbols(image, mode="stretch")), "default_stretch")
]

transformation_methods = [
    (transforms.Lambda(erode), "erode"),
    (transforms.Lambda(harsh_gamma_enhance), "gamma_enhance")
]

# Creates each combination of the pipeline where the transformation methods go first and are combined with each resizing method.
transformation_pipelines = []

for i in range(2):
    for k in range(len(transformation_methods) + 1):
        for t_subset in itertools.permutations(transformation_methods, k):
            for resize, resize_name in resizing_methods:
                pipeline = (
                    ([transforms.Lambda(resize_image)] if i == 0 else []) + 
                    list(map(lambda pair: pair[0], list(t_subset))) + 
                    [resize, transforms.Lambda(normalize), transforms.ToTensor()]
                )
                pipeline_name = "->".join(
                    ["resize"] if i == 0 else [] +
                    list(map(lambda pair: pair[1], list(t_subset))) +
                    [resize_name]
                )

                transformation_pipelines.append((transforms.Compose(pipeline), pipeline_name))

print(transformation_pipelines[0][0])

data = process_image(img=image_data, pipeline=transformation_pipelines[0][0])

print(data)

Compose(
    Lambda()
    Resize(size=(28, 28), interpolation=bilinear, max_size=None, antialias=True)
    Lambda()
    ToTensor()
)


In [4]:
from cnn import ConvNeuralNetwork
import torch.nn.functional as F
import torchvision.transforms as T

model = ConvNeuralNetwork(num_classes=17)
model.load_state_dict(torch.load("./model/model.pt"))
model.eval()

with torch.no_grad():
    outputs = model(images)

    probabilities = F.softmax(outputs, dim=1)
    print(probabilities)

    _, predicted = torch.max(outputs.data, 1)

    print(predicted)

    print((predicted == 14).sum().item())

str_values = [str(x) if x < 10 else label_decoding[int(x)] for x in predicted]

print(str_values)

tensor([[5.032e-10, 1.932e-07, 1.382e-06, 3.630e-05, 3.813e-08, 1.902e-06, 1.576e-09, 7.584e-10, 1.249e-07, 1.041e-06, 1.826e-09, 1.274e-05, 9.581e-01, 2.557e-08, 2.551e-06, 1.821e-06, 4.186e-02],
        [9.501e-07, 1.247e-07, 3.646e-06, 2.988e-07, 6.222e-05, 2.399e-05, 1.028e-04, 9.384e-06, 7.536e-04, 2.869e-06, 1.899e-05, 1.629e-11, 9.990e-01, 2.494e-07, 1.332e-07, 3.333e-08, 2.169e-06],
        [3.349e-09, 3.746e-09, 6.090e-07, 1.881e-05, 4.758e-11, 7.605e-08, 2.205e-12, 9.957e-09, 2.968e-09, 7.435e-06, 7.286e-11, 1.038e-05, 9.454e-01, 6.300e-10, 1.301e-05, 1.500e-07, 5.453e-02],
        [1.012e-08, 5.495e-10, 6.163e-06, 1.376e-07, 1.472e-11, 9.873e-08, 1.002e-11, 1.168e-05, 7.814e-11, 1.641e-10, 8.634e-11, 5.737e-11, 1.755e-09, 8.234e-13, 1.000e+00, 3.661e-10, 1.913e-08]])
tensor([12, 12, 12, 14])
1
['+', '+', '+', '=']


In [ ]:
import onnxruntime as ort
import numpy as np

MODEL_PATH = "../numpy_cnn/model/penman_cnn.onnx"

session = ort.InferenceSession(MODEL_PATH)

numpy_images = images.numpy()

output = session.run(None, {"X": numpy_images.astype(np.float32)})[0]
raw_values = list(np.argmax(output, axis=1))
str_values = [str(x) if x < 10 else label_decoding[x] for x in raw_values]

print(str_values)

['3', '8', '3', '3']


In [ ]:
pil_img = T.ToPILImage()

img = pil_img(images[0])
img.show()

print(images.numpy().shape)

(4, 1, 28, 28)
